**DataView - Exploração e Análise de Dados de Vendas**
Mini Projeto - Análise de Dados com Python

Importação das Bibliotecas

In [8]:
import pandas as pd
import numpy as np
import random
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

RF01 – Criar ou Carregar o Dataset de Vendas

In [9]:
def gerar_dataset_vendas(n_registros=150, seed=42):
    """Gera um dataset sintético de vendas com dados propositalmente sujos,
       incluindo valores nulos, strings sujas, datas inválidas e outliers.
    """
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor", "Teclado", "Mouse"]

    precos = {
        "Notebook": 3500,
        "Smartphone": 2200,
        "Tablet": 1800,
        "Monitor": 1200,
        "Teclado": 250,
        "Mouse": 120
    }

    categorias = {
        "Notebook": "Computadores",
        "Smartphone": "Celulares",
        "Tablet": "Celulares",
        "Monitor": "Computadores",
        "Teclado": "Periféricos",
        "Mouse": "Periféricos"
    }

    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    clientes = [f"Cliente_{i:03d}" for i in range(1, 31)]

    data_inicio = datetime(2024, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco = precos[produto]
        data = data_inicio + timedelta(days=random.randint(0, 364))

        if random.random() < 0.05:
            quantidade = None

        if random.random() < 0.04:
            preco = None

        if random.random() < 0.03:
            produto = " " + produto

        data_str = data.strftime("%Y-%m-%d") if random.random() > 0.02 else "DATA INVALIDA"

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_str,
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categorias.get(produto.strip(), "Outros"),
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })

    return pd.DataFrame(dados)


df_bruto = gerar_dataset_vendas()

os.makedirs("../data/raw", exist_ok=True)
df_bruto.to_csv("../data/raw/vendas.csv", index=False)

print(f"Dataset gerado com {len(df_bruto)} registros.")
display(df_bruto.head())

Dataset gerado com 150 registros.


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-01-13,Cliente_024,Mouse,Periféricos,Norte,2.0,120.0
1,2,2024-08-04,Cliente_018,Notebook,Computadores,Sul,NaN,3500.0
2,3,DATA INVALIDA,Cliente_026,Mouse,Periféricos,Sul,9.0,120.0
3,4,2024-06-23,Cliente_013,Mouse,Periféricos,Sudeste,7.0,120.0
4,5,2024-11-05,Cliente_030,Tablet,Celulares,Centro-Oeste,6.0,1800.0


RF02 – Inspecionar e Descrever os Dados

In [10]:
def inspecionar_dados(df):
    """ Exibe informações básicas do DataFrame."""
    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:")
    display(df.head())
    print(f"\nEstatísticas descritivas:")
    display(df.describe(include="all"))

    return df.describe(include="all")


df_bruto = pd.read_csv("../data/raw/vendas.csv")

descricao_inicial = inspecionar_dados(df_bruto)


=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (150, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        5
preco_unitario    2
dtype: int64

Primeiros registros:


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-01-13,Cliente_024,Mouse,Periféricos,Norte,2.0,120.0
1,2,2024-08-04,Cliente_018,Notebook,Computadores,Sul,NaN,3500.0
2,3,DATA INVALIDA,Cliente_026,Mouse,Periféricos,Sul,9.0,120.0
3,4,2024-06-23,Cliente_013,Mouse,Periféricos,Sudeste,7.0,120.0
4,5,2024-11-05,Cliente_030,Tablet,Celulares,Centro-Oeste,6.0,1800.0



Estatísticas descritivas:


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
count,150.000000,150,150,150,150,150,145.000000,148.000000
unique,NaN,117,30,8,3,5,NaN,NaN
top,NaN,DATA INVALIDA,Cliente_018,Mouse,Celulares,Sudeste,NaN,NaN
freq,NaN,4,8,28,51,41,NaN,NaN
mean,75.500000,NaN,NaN,NaN,NaN,NaN,5.468966,1558.513514
std,43.445368,NaN,NaN,NaN,NaN,NaN,2.808853,1190.199414
min,1.000000,NaN,NaN,NaN,NaN,NaN,1.000000,120.000000
25%,38.250000,NaN,NaN,NaN,NaN,NaN,3.000000,250.000000
50%,75.500000,NaN,NaN,NaN,NaN,NaN,5.000000,1800.000000
75%,112.750000,NaN,NaN,NaN,NaN,NaN,8.000000,2200.000000


RF03 – Limpar e Tratar os Dados

In [11]:
import os

def limpar_dados(df):
    df_limpo = df.copy()

    registros_iniciais = len(df_limpo)

    # Limpar espaços extras em colunas de texto
    colunas_texto = ["cliente", "produto", "categoria", "regiao"]

    for coluna in colunas_texto:
        df_limpo[coluna] = df_limpo[coluna].str.strip()

    # Converter datas inválidas para NaT
    df_limpo["data_venda"] = pd.to_datetime(
        df_limpo["data_venda"],
        errors="coerce"
    )

    datas_invalidas = df_limpo["data_venda"].isnull().sum()

    # Remover datas inválidas
    df_limpo = df_limpo.dropna(subset=["data_venda"])

    # Contar nulos antes de remover
    nulos_quantidade = df_limpo["quantidade"].isnull().sum()
    nulos_preco = df_limpo["preco_unitario"].isnull().sum()

    # Remover registros com nulos
    df_limpo = df_limpo.dropna(
        subset=["quantidade", "preco_unitario"]
    )

    registros_finais = len(df_limpo)

    print("=== RELATÓRIO DE LIMPEZA ===")
    print(f"Datas inválidas removidas: {datas_invalidas}")
    print(f"Nulos em quantidade removidos: {nulos_quantidade}")
    print(f"Nulos em preço removidos: {nulos_preco}")
    print(f"Registros iniciais: {registros_iniciais}")
    print(f"Registros finais: {registros_finais}")

    return df_limpo

In [12]:
df_limpo = limpar_dados(df_bruto)

=== RELATÓRIO DE LIMPEZA ===
Datas inválidas removidas: 4
Nulos em quantidade removidos: 4
Nulos em preço removidos: 2
Registros iniciais: 150
Registros finais: 140


In [14]:
os.makedirs("../data/processed/v1_com_outliers", exist_ok=True)

df_limpo.to_csv(
    "../data/processed/v1_com_outliers/vendas_v1.csv",
    index=False
)

print("Arquivo salvo em ../data/processed/v1_com_outliers/vendas_v1.csv")

Arquivo salvo em ../data/processed/v1_com_outliers/vendas_v1.csv


RF04 – Detectar e Tratar Outliers (versões v1 e v2)

RF05 – Criar Colunas Derivadas com Transformações

RF06 – Calcular Métricas Agregadas (groupby)

RF07 – Segmentar Clientes por Nível de Gasto

RF08 – Calcular Estatísticas com NumPy

RF09 – Criar Visualizações com Matplotlib e Seaborn

RF10 – Organizar o Código em Funções Reutilizáveis

RF11 – Ler e Escrever Arquivos (CSV e JSON)

RF12 – Consolidar a Análise e Salvar o Dataset Final